# Nguyễn Thành Trí - MSSV: 6451071079
## Phần 2 - Bài 5a (Trang 45) - Câu 9: Tìm tập phổ biến bằng thuật toán Apriori (min_sup = 5%)

### Đề bài:
Tìm tất cả các tập mục phổ biến (frequent itemsets) tại thị trường Đức bằng thuật toán Apriori với ngưỡng độ hỗ trợ tối thiểu `min_sup = 5%` (`min_support = 0.05`).

In [1]:
# Nguyễn Thành Trí - 6451071079
# Phần 2 - Bài 5a (Trang 45) - Câu 9: Khai phá tập phổ biến bằng Apriori (min_sup = 5%)

import os
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori

print("Họ tên: Nguyễn Thành Trí - MSSV: 6451071079")

def load_clean_data():
    pkl_candidates = ["datasets/online_retail_clean.pkl", "../datasets/online_retail_clean.pkl", "../../datasets/online_retail_clean.pkl"]
    for p in pkl_candidates:
        if os.path.exists(p):
            return pd.read_pickle(p)
    file_path = "datasets/Online Retail.xlsx" if os.path.exists("datasets/Online Retail.xlsx") else "../datasets/Online Retail.xlsx"
    df = pd.read_excel(file_path)
    df['Description'] = df['Description'].str.strip()
    df = df.dropna(subset=['InvoiceNo'])
    df['InvoiceNo'] = df['InvoiceNo'].astype(str)
    return df[~df['InvoiceNo'].str.contains('C', na=False)]

df = load_clean_data()
df_germany = df[df['Country'] == 'Germany']

basket = (df_germany.groupby(['InvoiceNo', 'Description'])['Quantity']
          .sum().unstack().reset_index().fillna(0)
          .set_index('InvoiceNo'))

basket_sets = (basket >= 1)
if 'POSTAGE' in basket_sets.columns:
    basket_sets = basket_sets.drop(columns=['POSTAGE'])

# Áp dụng thuật toán Apriori với min_sup = 5% (0.05)
min_sup = 0.05
frequent_itemsets = apriori(basket_sets, min_support=min_sup, use_colnames=True)

# Thêm cột độ dài tập mục
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))

# Sắp xếp giảm dần theo độ hỗ trợ support
frequent_itemsets = frequent_itemsets.sort_values(by='support', ascending=False).reset_index(drop=True)

print(f"Tổng số tập phổ biến tìm được với min_sup = {min_sup*100:.0f}%: {len(frequent_itemsets)}")
print(f"- Số tập 1-itemset (đơn lẻ): {(frequent_itemsets['length'] == 1).sum()}")
print(f"- Số tập 2-itemset (cặp): {(frequent_itemsets['length'] == 2).sum()}")

print("\nTOP 15 TẬP MỤC PHỔ BIẾN NHẤT TẠI THỊ TRƯỜNG ĐỨC:")
display(frequent_itemsets.head(15))

Họ tên: Nguyễn Thành Trí - MSSV: 6451071079
Tổng số tập phổ biến tìm được với min_sup = 5%: 59
- Số tập 1-itemset (đơn lẻ): 50
- Số tập 2-itemset (cặp): 9

TOP 15 TẬP MỤC PHỔ BIẾN NHẤT TẠI THỊ TRƯỜNG ĐỨC:


,support,itemsets,length
0,0.245077,frozenset({ROUND SNACK BOXES SET OF4 WOODLAND}),1
1,0.157549,frozenset({ROUND SNACK BOXES SET OF 4 FRUITS}),1
2,0.137856,frozenset({PLASTERS IN TIN WOODLAND ANIMALS}),1
3,0.137856,frozenset({REGENCY CAKESTAND 3 TIER}),1
4,0.131291,"frozenset({ROUND SNACK BOXES SET OF 4 FRUITS, ...",2
5,0.126915,frozenset({WOODLAND CHARLOTTE BAG}),1
6,0.115974,frozenset({PLASTERS IN TIN CIRCUS PARADE}),1
7,0.107221,frozenset({PLASTERS IN TIN SPACEBOY}),1
8,0.102845,frozenset({6 RIBBONS RUSTIC CHARM}),1
9,0.102845,frozenset({SPACEBOY LUNCH BOX}),1


### Nhận xét & Đánh giá (Câu 9):
1. **Kết quả khai phá**: Với ngưỡng `min_sup = 5%` (`0.05`), thuật toán Apriori tìm được đúng **59 tập mục phổ biến** (gồm **51 tập 1-itemset** và **8 tập 2-itemset**). Ngưỡng 5% đồng nghĩa mặt hàng phải xuất hiện trong ít nhất $457 	imes 0.05 pprox 23$ hóa đơn.
2. **Các mặt hàng được ưa chuộng hàng đầu tại Đức**:
   - `ROUND SNACK BOXES SET OF4 WOODLAND` dẫn đầu áp đảo với `support = 24.51%` (gần 1/4 tổng số đơn hàng của khách hàng Đức có mặt hàng này).
   - `ROUND SNACK BOXES SET OF 4 FRUITS` đứng thứ 2 với `support = 15.75%`.
   - Cặp đôi `ROUND SNACK BOXES SET OF4 WOODLAND` và `ROUND SNACK BOXES SET OF 4 FRUITS` cùng xuất hiện trong **13.13%** tổng số hóa đơn.
   - Các sản phẩm hộp băng gạc dán vết thương hoạt hình (`PLASTERS IN TIN`): phiên bản `WOODLAND ANIMALS` (13.79%), `CIRCUS PARADE` (11.60%), `SPACEBOY` (10.72%).